In [ ]:
import	torch
import	numpy					as	np
import	pandas					as	pd
import	base.arch_model			as	model
import	plotly.express			as	px
import	plotly.graph_objects	as	go
# how do I import this model?
ret_nums = 9
symb_name = "IBIT"
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_vailable() else "cpu")
nn_returns: model.feedforward_generation = model.feedforward_generation(device = device, input_size = ret_nums).to(device)
nn_returns.eval()
nn_returns.load_state_dict(torch.load(f'model/{symb_name}_{ret_nums}_model_params.pth'))
print("Finished loading model")
testing_pd: pd.DataFrame = pd.read_parquet(f'test/signed_returns_{ret_nums}/{symb_name}_returns_test.parquet')
x_pd: torch.Tensor = torch.tensor(testing_pd.drop(columns = ["pred_return"]).values, dtype = torch.float32, device = device)
y_pd: torch.Tensor = torch.tensor(testing_pd["pred_return"].values, dtype = torch.float32, device = device)
print("Finished loading data")

In [ ]:
output: torch.Tensor = nn_returns(x_pd)
mean: np.ndarray = output[:, 0].cpu().detach().numpy() * 50
var: np.ndarray = output[:, 1].cpu().detach().numpy()
mean

In [ ]:
returns_fig = go.Figure()
returns_fig.add_trace(go.Histogram(x = mean, histnorm = 'probability density', name = 'Predicted', opacity = 0.5))
returns_fig.add_trace(go.Histogram(x = y_pd.cpu().detach().numpy(), histnorm = 'probability density', name = 'Actual', opacity = 0.5))

In [ ]:
px.histogram(np.sqrt(var), histnorm = 'probability density')

In [ ]:
deviate = mean - y_pd.cpu().detach().numpy()
sq_dev = deviate * deviate
inner_error = sq_dev / (2 * var + 1e-7)
log_error = 1.5 * np.log(inner_error + 1 + 1e-7) + 0.5 * np.log(var)
print(log_error.mean())
px.histogram(log_error)

In [ ]:
import	scipy.stats	as	stats
gen_vals = stats.t.rvs(df = 2, size = log_error.shape[0])
sq_dev_gen = gen_vals * gen_vals
inner_error_gen = sq_dev_gen / 2
log_error_gen = 1.5 * np.log(inner_error_gen + 1)
px.histogram(log_error_gen)